# 09 · Corrective RAG (CRAG) with LangGraph
**Concept:** grade retrieved documents *before* trusting them; if they're weak, rewrite the query and fall back to another source, or answer INSUFFICIENT_DATA.
**Our adaptation:** the paper falls back to web search; we fall back to the **resolved-tickets** index (no extra key, stays inside our knowledge).

In [ ]:
from rag_common import MODEL, EMBED_MODEL, get_chat_model, get_vectorstore

model = get_chat_model()
vectorstore = get_vectorstore()          # opens the Chroma store built in Notebook 01
print(f"Chat: {MODEL} | Embeddings: {EMBED_MODEL} | chunks in store: {vectorstore._collection.count()}")

In [ ]:
from typing import Literal, TypedDict
from pydantic import BaseModel, Field
from langchain_core.documents import Document
from langgraph.graph import StateGraph, START, END
from rag_common import format_docs, doc_ids

class State(TypedDict, total=False):
    question: str
    query: str
    source: str                 # "kb" or "tickets"
    documents: list[Document]
    relevant: list[Document]
    path: list[str]             # for teaching: which nodes ran
    answer: str

class Grade(BaseModel):
    relevant: Literal["yes", "no"] = Field(description="Does the document help answer the question?")

grader = model.with_structured_output(Grade)
kb_ret = vectorstore.as_retriever(search_kwargs={"k": 4, "filter": {"doc_type": "kb"}})
ticket_ret = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 4, "filter": {"doc_type": "ticket"}})

## Nodes

In [ ]:
def retrieve_kb(s: State) -> State:
    return {"query": s["question"], "source": "kb", "documents": kb_ret.invoke(s["question"]), "path": ["retrieve_kb"]}

def grade(s: State) -> State:
    keep = [d for d in s["documents"]
            if grader.invoke(f"Question: {s['question']}\n\nDocument [{d.metadata['doc_id']}]:\n{d.page_content}").relevant == "yes"]
    print(f"   🧪 graded {s['source']}: {len(keep)}/{len(s['documents'])} relevant → {doc_ids(keep)}")
    return {"relevant": keep, "path": s["path"] + [f"grade_{s['source']}"]}

def rewrite(s: State) -> State:
    q = model.invoke(f"Rewrite this support question as a short search query describing the symptom. Reply with the query only.\n{s['question']}").content
    print(f"   ✏️  rewritten query: {q}")
    return {"query": q, "path": s["path"] + ["rewrite"]}

def retrieve_tickets(s: State) -> State:
    return {"source": "tickets", "documents": ticket_ret.invoke(s["query"]), "path": s["path"] + ["retrieve_tickets"]}

def generate(s: State) -> State:
    ans = model.invoke(f"Answer ONLY from the context and cite [doc_id].\n<context>\n{format_docs(s['relevant'])}\n</context>\n"
                       f"Question: {s['question']}").content
    return {"answer": ans, "path": s["path"] + ["generate"]}

def insufficient(s: State) -> State:
    return {"answer": "INSUFFICIENT_DATA: neither the KB nor resolved tickets cover this question.",
            "path": s["path"] + ["insufficient"]}

## Edges: the corrective decisions

In [ ]:
def after_grade(s: State) -> str:
    if s["relevant"]:
        return "generate"
    return "rewrite" if s["source"] == "kb" else "insufficient"

g = StateGraph(State)
for name, fn in [("retrieve_kb", retrieve_kb), ("grade", grade), ("rewrite", rewrite),
                 ("retrieve_tickets", retrieve_tickets), ("generate", generate), ("insufficient", insufficient)]:
    g.add_node(name, fn)
g.add_edge(START, "retrieve_kb")
g.add_edge("retrieve_kb", "grade")
g.add_conditional_edges("grade", after_grade, ["generate", "rewrite", "insufficient"])
g.add_edge("rewrite", "retrieve_tickets")
g.add_edge("retrieve_tickets", "grade")
g.add_edge("generate", END)
g.add_edge("insufficient", END)
crag = g.compile()
print(crag.get_graph().draw_mermaid())

## Run it

In [ ]:
def run(q: str):
    print(f"\n❓ {q}")
    out = crag.invoke({"question": q})
    print("   path:", " → ".join(out["path"]))
    print("💬", out["answer"])

run("Our site-to-site tunnels on 5.1.3 reset every time phase 2 rekeys, but only towards one partner. Why?")   # KB path
run("Has a customer ever fixed a hanging orchestrator policy push by splitting the route-map?")             # may fall back to tickets
run("When will the Orion OW-20 support Wi-Fi 8?")                                                         # nothing anywhere

---
**🔧 TAC takeaway:** QA-check the KB hit before quoting it. If it doesn't fit, search the case archive; if that fails too, say so honestly. CRAG turns silent failures into visible ones.